In [ ]:
# %%capture
# !pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1
# !pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## IMPORTANT: Make sure to restart the session after installing the packages above.

In [ ]:
ZHIPU_APIKEY = ""

# 零、总述

Search(搜索) 是最早广泛应用于行业的大语言模型应用之一。在具有里程碑意义发布的论文 BERT【Pre-training of deep bidirectional transformers for language understanding】（2018年发布） 后的几个月, Google 使用了此方法来支持搜索，并且 Google 表示：它是 Search 历史上的一大里程碑式的飞跃。与此同时，Microsoft Bing 也不甘落后，它表示：“从今年 4 月开始，我们使用大小 transform 模型，为 Bing 用户带来了过去一年中幅度最大的质量提升。”

它们带来的这种能力被称为 **语义搜索（Semantic Search）**，它使搜索能够基于文本的含义进行，而不仅仅依靠关键词匹配。

与此同时，文本生成模型的快速普及使许多用户开始向模型提问，并期待模型能够给出事实性的答案。虽然这些模型能够流畅且自信地回答问题，但它们给出的答案并不总是正确的，也不一定是最新的。这个问题被称为**幻觉（Hallucinations）**。目前减少幻觉的一种主要方法，是构建能够检索相关信息的系统，并将检索到的信息提供给 LLM，从而帮助模型生成更符合事实的答案。终止方法被称为 **RAG（Retrieval-Augmented Generation）**

下面介绍三个概念：Dense Retrieval，Reranking, 和 RAG，这些概念不仅仅应用在 LLM 的搜索域中。



1. **Dense Retrieval(Dense 检索)**

Dense 检索的系统依赖于 Embedding 的概念，它会将搜索的问题转换成检索离此问题最近邻的搜索上。如下图所示：

<center>
<img src="./resources/dense_retrieval.png">
</center>

2. **Reranking(重排)**

搜索通常包含多步，重排序的语言模型通常是这些步骤之一，它的目标是计算代表检索出的文档与查询的相关性的一个分数，之后它就会按照这个分数进行重排。如下图所示：

<center>
<img src="./resources/reranking.png">
</center>


3. **RAG**

生成式搜索（Generative Search）是一个更广泛的一类子集，我们通常叫他 RAG。文本生成系统会包含搜索能力以减少幻觉，提升事实的准确性，还可以让生成模型基于指定的数据集生成内容。RAG 的流程如下图所示：

<center>
<img src="./resources/rag.png">
</center>


## 一、使用语言模型进行语义搜索

### 1.1 Dense 检索

Embedding 会将文本转换成空间的某个点，它们的距离越近，表示两个文本越相似。想象这样一个场景，用户在问题的时候，我们会将问题嵌入到与文本库相同的空间中，之后，我们去找与它最近的几个文档即可。如下图所示：query 代表问题，text n 代表文档

<center>
<img src="./resources/relevant.png">
</center>

可以看到，“text 2” 是问题 query 最好的回答，接下来是 "text 1"。那么这就会出现两个问题：

- text 3 是否应该作为结果返回呢？这个取决于你。有时，我们会设定一个很大的**阈值**来将不太相关的内容纳入进来
- 问题 query 和最好的回答语义相似么？并不总是这样。这就是为什么语言模型需要在 QA 对上反复的训练来变得更优秀

#### 1.1.1 示例

让我们基于智谱官网的模型介绍，来探索关于 GLM-5.3 的内容

我们分为以下四步：

1. 我们会将相关文档切成每一句来让文本变得可搜索
2. 将每一句嵌入到向量空间中
3. 构造搜索索引
4. 搜索并查看结果

##### 1. Getting the text archive and chunking it

我们只使用其中的一部分，完整的文章在 [GLM 5.3介绍](https://docs.bigmodel.cn/cn/guide/models/text/glm-5.3)

In [ ]:
text = """
在 GLM-5.3 的训练中，我们进一步推进了任务环境的规模化建设，使训练任务不再只是传统的编程题，而是更接近专家在真实工作中承担的完整专业工作单元。这些环境覆盖了更广泛的生产级工作流，任务设计也更加贴近工程与研究工作的实际开展方式，其中部分任务即使由经验丰富的工程师完成，也需要数天时间。
例如，在一项机器学习基础设施任务中，模型可能会获得与工程师相同的工作环境，包括计算集群、存储系统、内部文档、代码库和实验结果等资源。模型需要定位训练技术栈中的性能瓶颈，实施优化方案，运行实验，并在保证结果正确性的前提下，实现可量化的端到端加速。通过在此类高复杂度环境中训练，推动模型逐步具备端到端负责并完成复杂工作的能力，而不再依赖用户拆解问题并逐步监督执行。
随着智能体能力提升，后训练规模化的主要难点也逐渐从模型本身转移到任务环境。一个真正有价值的任务环境必须可执行、可验证，并贴近真实的专业工作；同时，我们需要的不是少量人工构建的环境，而是大规模、多样化的任务环境。为实现这一过程的规模化，我们构建了能够端到端合成任务环境的流水线，并针对部分任务进一步自动生成强化学习所需的奖励信号。研究智能体从真实工作中收集任务模式，并将其转化为可运行的长程任务环境，其中包含多步骤依赖关系和隐藏状态；随后，评审智能体会实际尝试完成每项任务，以验证任务是否确实可解。验证器在不访问参考解法的前提下生成；与此同时，系统会利用求解轨迹识别并消除奖励捷径。只有同时通过 Oracle、空操作和未解决状态检查的验证器，其生成的二值奖励信号才足够可靠，可直接用于模型训练。
GLM-5.3 延续了 GLM-5.2 中引入的强化学习策略，包括结合上下文压缩机制的 SAO，使能力增益能够在长程任务中持续体现，而非仅局限于短程任务。这些提升同时反映在编程任务和通用智能体任务中：GLM-5.3 在 Terminal-Bench 3.0 上的得分由 4.6 提升至 28.3，在 DeepSWE v1.1 上由 46.2 提升至 66.9，在 Agents’ Last Exam 上由 23.8 提升至 28.5。目前，这些流水线仍需要较多人工参与。进一步提升任务环境生成与验证过程的自主化程度，是我们下一阶段的重点方向之一。
除公开基准测试外，我们还推出了内部基准 Z.ai Code Bench，用于在贴近真实用户场景的条件下评估编程智能体。该基准覆盖多种任务类别，并将智能体置于复杂的本地开发环境中。在不同推理强度档位下，我们从两个维度评估智能体：一是端到端任务完成率，二是细粒度检查项准确率。作为一项非公开基准，Z.ai Code Bench 还能降低公开测试集污染带来的风险，从而更准确地反映模型在真实用户场景中的使用体验。"""

# 使用句号将文本转成 list
texts = text.split('。')

# 去空格并加换行
texts = [t.strip(' \n') for t in texts]

##### 2. Embedding the Text Chunks

In [ ]:
import requests
import numpy as np

def embed_text(docs, model="embedding-3", embed_dim=2048):
    """
    将 docs 使用 embedding-3 嵌入之后转成 embedding 向量
    """

    url = "https://open.bigmodel.cn/api/paas/v4/embeddings"

    payload = {
        "model": model,
        "input": docs,
        "dimensions": embed_dim
    }
    headers = {
        "Authorization": f"Bearer {ZHIPU_APIKEY}",
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)

    print("嵌入模型成功，结果为：")
    print(response.text)

    # 获取 embedding 向量
    data = response.text["data"]

    embeds = np.array(
        [item["embedding"] for item in data]
    )

    return embeds

将我们的文档嵌入

In [ ]:
embeds = embed_text(texts)
print(embeds.shape)

##### 3. Building The Search Index

在搜索之前，我们需要创建搜索索引来加快检索的速度。向量的索引与传统数据库的索引不同，传统数据库索引实现的常见技术是B+树和 Hash，而向量的索引使用的是 Flat、IVF 和 HNSW 等技术。其中：

- **Flat(暴力搜索)**：遍历所有向量，逐一计算距离
- **IVF（Inverted File Index, 倒排文件索引）**：先对向量聚类，查询时只搜索部分相关的聚类
- **HNSW（Hierarchical Navigable Small World, 分层导航小世界图）**: 将向量组织成多层图，通过图上的邻居关系逐步接近目标向量


我们这里使用 FAISS 来加快搜索, 它负责存储、索引和搜索向量，并且它实现了上面三种技术。它有以下特点：

- 支持高维向量相似度搜索
- 支持欧氏距离、内积等度量方式
- 提供精确最近邻搜索和近似最近邻搜索
- 底层主要使用 C++ 实现，提供 Python 接口
- 部分索引和搜索算法支持 GPU 加速

我们来说一下 FAISS 的索引类型，如下表所示

| 索引类型            | 说明                                                     | 是否需要训练 |
|-----------------|--------------------------------------------------------|---|
| `IndexFlatL2`   | 基于欧氏距离（L2）的精确最近邻搜索，返回距离最近的 K 个向量                       | 不需要 |
| `IndexFlatIP`   | 基于内积（Inner Product）的精确最近邻搜索，返回内积最大的 K 个向量              | 不需要 |
| `IndexIVFFlat`  | 基于倒排文件的近似最近邻搜索，通过 K-Means 将向量划分到多个聚类中，搜索时只访问部分聚类，减少计算量 | 需要 |
| `IndexIVFPQ`    | 结合倒排文件和乘积量化（PQ）的近似最近邻搜索。先通过聚类缩小搜索范围，再利用 PQ 对向量进行压缩     | 需要 |
| `IndexHNSWFlat` | 基于 HNSW 的近似最近邻搜索，通过构建多层邻居图，在图中逐步寻找与查询向量最接近的节点。         | 不需要 |

⚠️⚠️⚠️**注意，这里的训练不是训练 Embedding 模型，而是说** <span style="color: red">**有些 FAISS 索引需要通过训练来学习向量的聚类中心或量化参数，从而建立更高效的搜索结构**</span>


下面我们为嵌入到 embedding-3 模型的向量创建搜索索引，这里使用 `IndexFlatL2` 来创建

In [ ]:
import faiss

dim = embeds.shape[1]

# 创建指定维度的索引
index = faiss.IndexFlatL2(dim)

# 将向量添加到索引中
index.add(np.float32(embeds))

让我们看看 `IndexFlatL2` 是否可训练？

In [ ]:
print(index.is_trained)

##### 4. Search the Index

我们现在可以用我们想去 query 的东西搜索数据集了，让我们先来定义一个 search 函数

In [ ]:
import pandas as pd

def search(query, number_of_results=3):

    # 1. 获取查询向量的嵌入
    query_embed = embed_text(query)

    # 2. 检索最近邻的向量: 找3个
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

    # 3. 格式化结果：只选第一个
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                                 'distance': distances[0]})

    # 4. 输入结果
    print(f"与 {query} 最近邻的是：")
    return results

In [ ]:
query = "GLM 5.3 相比 GLM 5.2 强在什么地方？"
results = search(query)

In [ ]:
results

我们可以比较传统的关键字搜索和上述的语义搜索，在这里，我们使用 BM25 算法，它是一种词汇搜索的算法

In [ ]:
import tqdm
from rank_bm25 import BM250kapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)

    return tokenized_doc

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM250kapi(tokenized_corpus)

定义关键字检索函数

In [ ]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("输入的问题为：", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print("Top 3 BM25 Hits:")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

In [ ]:
keyword_search(query)

可以看到，查询出来的第一个结果并不是我问题的答案，虽然他们都有 `GLM 5.3` 这个词。

#### 1.1.2 Dense 检索的局限性（Caveats of dense retrieval）

我们需要了解 Dense 检索的缺点并且要知道怎么处理它们。如果搜索出来的文本没有包含答案会发生什么呢？举个例子来说

In [ ]:
query = "地球的质量是多少？"
results = search(query)

In [ ]:
results

为了解决这种现象，我们可以增加它们的阈值

Dense 检索的另一个缺点：用户如果想了解这个问题会精确匹配哪个段落，该怎么办？这就是关键字检索的好处了。解决此问题的方法是我们要使用混合检索（Hybrid Search）,它包含语义检索和关键词检索。

Dense 检索有一个域的限制，举个例子来说，如果你在维基百科的数据上训练检索模型，那么将其部署到法律领域上，这个模型表现得不会很好。

最后一个要说明的是，如果检索的答案分布在多条信息中，我们该怎么办呢。这就要考虑到 Dense 检索系统设计的重要性了：长文本切块的最好方式是什么？为什么我们要先进行切块？

#### 1.1.3 改善 Dense 检索

那么，我们怎么改善 Dense 检索的质量呢？下面介绍两个方法

- 长文本切块策略
- 微调 Embedding 模型

##### 1.1.3.1 长文本切块策略

由于大模型上下文长度的限制，我们不能将全部的文本一次喂给模型，所以我们要对长文本进行切块，下面是切块的策略

| 切块策略 | 英文名称 | 原理 | 优点 | 缺点 |
|---|---|---|---|---|
| 固定长度切块 | Fixed-size Chunking | 按固定字符数或 Token 数切分 | 简单、高效 | 容易破坏语义 |
| 滑动窗口切块 | Sliding Window Chunking | 固定长度切块，相邻 Chunk 保留重叠内容 | 减少上下文丢失 | 存在文本冗余 |
| 递归切块 | Recursive Chunking | 按段落、换行、句子等分隔符逐级切分 | 尽可能保持文本结构 | 无法完全保证语义完整 |
| 句子切块 | Sentence-based Chunking | 按句子边界切分，并组合相邻句子 | 句子完整 | Chunk 长度不均匀 |
| 语义切块 | Semantic Chunking | 根据 Embedding 的语义相似度确定切分位置 | 语义连续性好 | 计算成本较高 |
| 文档结构切块 | Document-based Chunking | 根据标题、章节、表格、代码块等结构切分 | 保留文档结构 | 依赖文档格式 |
| 层次化切块 | Hierarchical Chunking | 将文档组织为父 Chunk 和子 Chunk | 兼顾检索精度与上下文 | 实现较复杂 |
| LLM 智能切块 | LLM-based Chunking | 利用 LLM 判断主题及语义边界 | 适合复杂文档 | 成本高、速度慢 |

##### 1.1.3.2 微调嵌入模型

Dense 检索优化的更多的是要优化文本的嵌入，而不仅仅是token的嵌入。

我们来看这样一个例子，句子 “Interstellar（星际穿越）于2014年10月26日在洛杉矶首映.”。与此句子相关的查询是：

- Interstellar 的上映日期？
- Interstellar 什么时候首映？

微调过程就是让这些查询的嵌入向量更接近与目标句。它也需要看到对于查询的负样本（它与句子毫不相关），负样本为：Interstellar 的演员阵容。

我们现在有三个样本对了，两个正样本和一个负样本。让我们假设，这三个查询都与原来的文档有相同的距离，之后我们将进行**微调**：*让正样本离的更近，让负样本离得更远*。如下图所示

<center>
<img src="./resources/fine_tuning.png">
</center>

### 1.2 重排名（Reranking）

为了改善检索质量，我们将搜索结果进行重排名，如下图所示，重排序在第二阶段

<center>
<img src="./resources/reranking_two_stage.png">
</center>

In [ ]:
import requests

url = "https://open.bigmodel.cn/api/paas/v4/rerank"

payload = {
    "model": "rerank",
    "query": "查询候选文本A",
    "documents": ["需要打分的候选文本A", "需要打分的候选文本B"],
    "top_n": 4
}
headers = {
    "Authorization": f"Bearer {ZHIPU_APIKEY}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.text)

### 1.3 检索的评估指标

## 二、RAG（Retrieval-Augmented-Generation）

### 2.1 从语义搜索到 RAG

### 2.2 高级的 RAG 技术

### 2.3 RAG 评估